In [ ]:
# Install required packages
!pip install numpy librosa scikit-learn xgboost tqdm -q

: 

In [ ]:
# Imports
import numpy as np
import librosa
from pathlib import Path
import pickle
import json
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score
import xgboost as xgb

print("✅ All imports successful!")

In [ ]:
# Configuration
DATASET_PATH = Path("dataset") / "Raw JL corpus (unchecked and unannotated)" / "JL(wav+txt)"
MODEL_SAVE_PATH = Path("models")
MODEL_SAVE_PATH.mkdir(exist_ok=True)

SAMPLE_RATE = 22050
BATCH_SIZE = 50

# Emotion to stress mapping
EMOTION_MAP = {
    'neutral': 10, 'happy': 15, 'encouraging': 18, 'assertive': 35,
    'excited': 50, 'apologetic': 58, 'sad': 68, 'concerned': 72,
}

print(f"Dataset path: {DATASET_PATH}")
print(f"Dataset exists: {DATASET_PATH.exists()}")

In [ ]:
# Feature extraction function
def extract_features(audio_path):
    """Extract 55 audio features from file."""
    try:
        audio, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
        if len(audio) < sr * 0.3:
            return None
        
        audio = librosa.util.normalize(audio)
        features = []
        
        # MFCC (26 features)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
        features.extend(np.mean(mfcc, axis=1).flatten().tolist())
        features.extend(np.std(mfcc, axis=1).flatten().tolist())
        
        # Spectral (10 features)
        features.append(float(np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr))))
        features.append(float(np.std(librosa.feature.spectral_centroid(y=audio, sr=sr))))
        features.append(float(np.mean(librosa.feature.spectral_rolloff(y=audio, sr=sr))))
        features.append(float(np.std(librosa.feature.spectral_rolloff(y=audio, sr=sr))))
        features.append(float(np.mean(librosa.feature.spectral_bandwidth(y=audio, sr=sr))))
        features.append(float(np.std(librosa.feature.spectral_bandwidth(y=audio, sr=sr))))
        features.append(float(np.mean(librosa.feature.spectral_contrast(y=audio, sr=sr))))
        features.append(float(np.std(librosa.feature.spectral_contrast(y=audio, sr=sr))))
        features.append(float(np.mean(librosa.feature.spectral_flatness(y=audio))))
        features.append(float(np.std(librosa.feature.spectral_flatness(y=audio))))
        
        # Chroma, ZCR, RMS (8 features)
        chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
        features.append(float(np.mean(chroma)))
        features.append(float(np.std(chroma)))
        
        zcr = librosa.feature.zero_crossing_rate(audio)
        features.append(float(np.mean(zcr)))
        features.append(float(np.std(zcr)))
        
        rms = librosa.feature.rms(y=audio)
        features.append(float(np.mean(rms)))
        features.append(float(np.std(rms)))
        features.append(float(np.max(rms)))
        features.append(float(np.min(rms)))
        
        # Pitch (6 features)
        f0, _, _ = librosa.pyin(audio, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'), sr=sr)
        f0_clean = f0[~np.isnan(f0)]
        if len(f0_clean) > 1:
            features.extend([
                float(np.mean(f0_clean)), float(np.std(f0_clean)),
                float(np.max(f0_clean)), float(np.min(f0_clean)),
                float(np.ptp(f0_clean)), float(np.mean(np.abs(np.diff(f0_clean))))
            ])
        else:
            features.extend([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
        
        # Tempo (2 features)
        tempo, beats = librosa.beat.beat_track(y=audio, sr=sr)
        features.append(float(tempo))
        features.append(float(len(beats) / (len(audio) / sr)))
        
        # Prosodic (3 features)
        features.append(float(np.var(rms[0])))
        features.append(float(np.var(f0_clean)) if len(f0_clean) > 0 else 0.0)
        features.append(float(np.sum(zcr) / (len(audio) / sr)))
        
        return np.array(features)
    except:
        return None

print("✅ Feature extraction function defined")

In [ ]:
# Load dataset
audio_files = list(DATASET_PATH.glob("*.wav"))
print(f"Found {len(audio_files)} audio files")

X_list, y_list = [], []
emotion_counts = {}

print("\nExtracting features (15-20 minutes)...\n")

for i in tqdm(range(0, len(audio_files), BATCH_SIZE), desc="Processing"):
    for audio_file in audio_files[i:i+BATCH_SIZE]:
        emotion = audio_file.stem.split('_')[1] if len(audio_file.stem.split('_')) >= 2 else None
        if emotion not in EMOTION_MAP:
            continue
        
        features = extract_features(audio_file)
        if features is not None:
            X_list.append(features)
            y_list.append(emotion)
            emotion_counts[emotion] = emotion_counts.get(emotion, 0) + 1

X = np.array(X_list)

print(f"\n✅ Extracted {len(X)} samples with {X.shape[1]} features")
print(f"\nEmotion distribution:")
for emotion, count in sorted(emotion_counts.items(), key=lambda x: EMOTION_MAP[x[0]]):
    print(f"  {emotion:12s} ({EMOTION_MAP[emotion]:2d}): {count:4d} samples")

In [ ]:
# Convert emotions to numeric labels for classification
emotion_to_id = {e: i for i, e in enumerate(sorted(EMOTION_MAP.keys(), key=lambda x: EMOTION_MAP[x]))}
id_to_emotion = {i: e for e, i in emotion_to_id.items()}

y_numeric = np.array([emotion_to_id[emotion] for emotion in y_list])

print("Emotion to ID mapping:")
for emotion, id in emotion_to_id.items():
    print(f"  {emotion:12s} → {id} (stress: {EMOTION_MAP[emotion]})")

In [ ]:
# Split and normalize data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_numeric, test_size=0.2, random_state=42, stratify=y_numeric
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✅ Data prepared for training")

In [ ]:
# Train XGBoost emotion classifier
print("Training XGBoost emotion classifier...\n")

model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    gamma=0.1,
    random_state=42,
    tree_method='hist',
    device='cpu',
    num_class=len(emotion_to_id)
)

model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=True
)

print("\n✅ Training complete!")

In [ ]:
# Evaluate emotion classification
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

train_accuracy = accuracy_score(y_train, y_train_pred) * 100
test_accuracy = accuracy_score(y_test, y_test_pred) * 100

print("="*80)
print("EMOTION CLASSIFICATION RESULTS")
print("="*80)
print(f"\nTrain Accuracy: {train_accuracy:.1f}%")
print(f"Test Accuracy:  {test_accuracy:.1f}%")

# Convert to stress scores
y_test_stress_true = np.array([EMOTION_MAP[id_to_emotion[id]] for id in y_test])
y_test_stress_pred = np.array([EMOTION_MAP[id_to_emotion[id]] for id in y_test_pred])

stress_mae = mean_absolute_error(y_test_stress_true, y_test_stress_pred)
stress_r2 = r2_score(y_test_stress_true, y_test_stress_pred)

print(f"\nSTRESS DETECTION PERFORMANCE:")
print(f"MAE:  {stress_mae:.2f} points")
print(f"R²:   {stress_r2:.4f}")

if test_accuracy >= 70:
    print(f"\n🎉 SUCCESS! {test_accuracy:.1f}% accuracy")
elif test_accuracy >= 60:
    print(f"\n✅ Good: {test_accuracy:.1f}%")
else:
    print(f"\n⚠️ {test_accuracy:.1f}% - needs improvement")

In [ ]:
# Save model
with open(MODEL_SAVE_PATH / "emotion_classifier.pkl", 'wb') as f:
    pickle.dump(model, f)
print("✅ Model saved")

with open(MODEL_SAVE_PATH / "scaler.pkl", 'wb') as f:
    pickle.dump(scaler, f)
print("✅ Scaler saved")

metrics = {
    'emotion_train_accuracy': float(train_accuracy),
    'emotion_test_accuracy': float(test_accuracy),
    'stress_mae': float(stress_mae),
    'stress_r2': float(stress_r2),
    'emotion_to_id': emotion_to_id,
    'emotion_to_stress': EMOTION_MAP
}

with open(MODEL_SAVE_PATH / "metrics.json", 'w') as f:
    json.dump(metrics, f, indent=4)
print("✅ Metrics saved")

print(f"\n📦 All saved to: {MODEL_SAVE_PATH}")